# Раздел 1. Основы теории вероятностей
---

## Тема 4. Повторные независимые испытания
---

In [1]:
import numpy as np
import scipy.stats as st

### Справка: что считает `scipy.stats`

Для $n$ независимых испытаний с вероятностью успеха $p$ (схема Бернулли) и числа успехов $m$:

* `st.binom.pmf(m, n, p)` – формула Бернулли $P_{m,n} = C_n^m p^m q^{n-m}$;
* `st.binom.cdf(m, n, p)` – вероятность $P(X \leqslant m)$ (сумма $P_{0,n} + \ldots + P_{m,n}$);
* `st.binom.sf(m, n, p)` – вероятность $P(X > m) = 1 - P(X \leqslant m)$;
* `st.poisson.pmf(m, lam)` – формула Пуассона $P_m(\lambda) = \frac{\lambda^m e^{-\lambda}}{m!}$, $\lambda = np$;
* `st.norm.pdf(x)` – функция Гаусса $f(x) = \frac{1}{\sqrt{2\pi}} e^{-x^2/2}$ из локальной формулы Муавра–Лапласа;
* `st.norm.cdf(x)` – функция стандартного нормального распределения. С функцией Лапласа из учебника она связана так: $\Phi(x) = 2\,\texttt{norm.cdf}(x) - 1$, поэтому интегральная формула $P_n(a \leqslant m \leqslant b) \approx \frac{1}{2}\left[ \Phi(x_2) - \Phi(x_1) \right]$ превращается в `norm.cdf(x2) - norm.cdf(x1)`.

Таблицы приложений учебника при этом не нужны.

## Пример 2.9
> В среднем $20\%$ пакетов акций на аукционах продаются по первоначально заявленной цене. Найти вероятность того, что из $9$ пакетов акций в результате торгов по первоначально заявленной цене: 1) не будут проданы $5$ пакетов; 2) будет продано: а) менее $2$ пакетов; б) не более $2$ пакетов; в) хотя бы $2$ пакета; г) наивероятнейшее число пакетов.

Торги по каждому пакету – независимые испытания с двумя исходами, поэтому работаем по **формуле Бернулли**:

$P_{m,n} = C_n^m p^m q^{n-m}$

Число испытаний $n = 9$ мало, поэтому приближённые формулы здесь не годятся (см. замечание в конце задачи).

In [2]:
n = 9       # число пакетов акций
p = 0.2     # вероятность продажи по заявленной цене
q = 1 - p

* Рассмотрим п. **1)** – **не** продано ровно $5$ пакетов

Для события «не продан» вероятность равна $q = 0{,}8$, т.е. нужно найти $P_{5,9}$ при $p' = 0{,}8$:

$P_{5,9} = C_9^5 \cdot 0{,}8^5 \cdot 0{,}2^4$

In [3]:
st.binom.pmf(5, n, q)

np.float64(0.06606028799999995)

* Рассмотрим п. **2а)** – продано **менее** $2$ пакетов

Событие «$m < 2$» – это «$m = 0$» или «$m = 1$» (несовместные):

$P_9(m < 2) = P_{0,9} + P_{1,9} = C_9^0 \cdot 0{,}2^0 \cdot 0{,}8^9 + C_9^1 \cdot 0{,}2 \cdot 0{,}8^8$

Сумма вероятностей до $m$ включительно – это функция `cdf`, поэтому $P(m < 2) = P(m \leqslant 1)$.

In [4]:
st.binom.cdf(1, n, p)

np.float64(0.4362076159999997)

* Рассмотрим п. **2б)** – продано **не более** $2$ пакетов

$P_9(m \leqslant 2) = P_{0,9} + P_{1,9} + P_{2,9}$

In [5]:
st.binom.cdf(2, n, p)

np.float64(0.7381975039999998)

* Рассмотрим п. **2в)** – продано **хотя бы** $2$ пакета

Прямой путь требует суммы $P_{2,9} + P_{3,9} + \ldots + P_{9,9}$, поэтому переходим к противоположному событию «$m < 2$»:

$P_9(m \geqslant 2) = 1 - P_9(m < 2) = 1 - \left( P_{0,9} + P_{1,9} \right)$

В `scipy` это функция `sf` (survival function): `sf(1)` $= P(X > 1)$.

In [6]:
round(1 - st.binom.cdf(1, n, p), 4)

np.float64(0.5638)

In [7]:
round(st.binom.sf(1, n, p), 4)   # то же самое через sf

np.float64(0.5638)

* Рассмотрим п. **2г)** – **наивероятнейшее** число проданных пакетов

Наивероятнейшее число $m_0$ определяется двойным неравенством **(2.4)**:

$np - q \leqslant m_0 \leqslant np + p$

$9 \cdot 0{,}2 - 0{,}8 \leqslant m_0 \leqslant 9 \cdot 0{,}2 + 0{,}2 \quad \Rightarrow \quad 1 \leqslant m_0 \leqslant 2$

Так как $np = 1{,}8$ не целое, а длина промежутка равна $p + q = 1$, то целых чисел в нём два: $m_0 = 1$ и $m_0' = 2$. Вероятность каждого из них находим по формуле Бернулли.

In [8]:
lo = round(n * p - q, 9)   # границы двойного неравенства (2.4)
hi = round(n * p + p, 9)
m0 = [k for k in range(n + 1) if lo <= k <= hi]
print("наивероятнейшие числа:", m0)

for k in m0:
    print(f"P_{k},{n} =", round(st.binom.pmf(k, n, p), 4))

round(sum(st.binom.pmf(k, n, p) for k in m0), 4)

наивероятнейшие числа: [1, 2]
P_1,9 = 0.302
P_2,9 = 0.302


np.float64(0.604)

**Проверка.** Посмотрим на весь ряд распределения – максимум действительно достигается при $m = 1$ и $m = 2$.

In [9]:
pmf = st.binom.pmf(np.arange(n + 1), n, p)
for m, pm in enumerate(pmf):
    print(f"m = {m}:  P = {pm:.4f}")

m = 0:  P = 0.1342
m = 1:  P = 0.3020
m = 2:  P = 0.3020
m = 3:  P = 0.1762
m = 4:  P = 0.0661
m = 5:  P = 0.0165
m = 6:  P = 0.0028
m = 7:  P = 0.0003
m = 8:  P = 0.0000
m = 9:  P = 0.0000


> **Замечание.** Приближённые формулы здесь неприменимы: $npq = 9 \cdot 0{,}2 \cdot 0{,}8 = 1{,}44 < 20$ (условие Муавра–Лапласа), а формула Пуассона требует большого $n$. Для сравнения:

In [10]:
print("точно   P(m < 2):", round(st.binom.cdf(1, n, p), 4))
print("Пуассон P(m < 2):", round(st.poisson.cdf(1, n * p), 4))

точно   P(m < 2): 0.4362
Пуассон P(m < 2): 0.4628


Расхождение уже во втором знаке – поэтому берём точную формулу Бернулли.

**Ответ:** 1) $0{,}066$; 2) а) $0{,}436$; б) $0{,}738$; в) $0{,}564$; г) $m_0 = 1$ и $m_0' = 2$, $P_{1,9} = P_{2,9} \approx 0{,}302$ (вероятность реализации одного из них $\approx 0{,}604$).

## Пример 2.10
> Завод отправил на базу $10\,000$ стандартных изделий. Среднее число изделий, повреждаемых при транспортировке, составляет $0{,}02\%$. Найти вероятность того, что из $10\,000$ изделий: 1) будет повреждено: а) $3$; б) по крайней мере $3$; 2) не будет повреждено: а) $9997$; б) хотя бы $9997$.

Вероятность повреждения одного изделия $p = 0{,}0002$ **мала**, число изделий $n = 10\,000$ **велико**, а $\lambda = np = 2 \leqslant 10$ – выполнены условия **формулы Пуассона (2.6)**:

$P_{m,n} \approx \frac{\lambda^m e^{-\lambda}}{m!}$

In [11]:
n = 10000      # число изделий
p = 0.0002     # вероятность повреждения одного изделия
lam = n * p    # параметр Пуассона
lam

2.0

* Рассмотрим п. **1а)** – повреждено ровно $3$ изделия

$P_{3,10000} \approx \frac{2^3 e^{-2}}{3!}$

In [12]:
round(st.poisson.pmf(3, lam), 4)

np.float64(0.1804)

Для сравнения – точное значение по формуле Бернулли:

In [13]:
round(st.binom.pmf(3, n, p), 4)

np.float64(0.1805)

* Рассмотрим п. **1б)** – повреждено **по крайней мере** $3$ изделия

Прямое суммирование $P_{3,n} + P_{4,n} + \ldots + P_{n,n}$ громоздко, поэтому переходим к противоположному событию:

$P_{10000}(m \geqslant 3) = 1 - P_{10000}(m < 3) = 1 - \left( P_{0,10000} + P_{1,10000} + P_{2,10000} \right)$

In [14]:
round(1 - st.poisson.cdf(2, lam), 4)

np.float64(0.3233)

> **Замечание.** Интегральную формулу Муавра–Лапласа здесь применять нельзя: $npq \approx 2 < 20$.

* Рассмотрим п. **2а)** – **не** повреждено ровно $9997$ изделий

Теперь «успех» – целое изделие, $p' = 1 - 0{,}0002 = 0{,}9998$ **близко к единице**, поэтому ни формула Пуассона, ни локальная формула Муавра–Лапласа ($npq \approx 2 < 20$) напрямую не подходят. Но событие «не повреждено $9997$ из $10\,000$» равносильно событию «повреждено $3$ из $10\,000$», т.е. задача сводится к п. 1а):

$P_{9997,10000} = P_{3,10000} \approx 0{,}1804$

In [15]:
round(st.poisson.pmf(3, lam), 4)

np.float64(0.1804)

Точная проверка по формуле Бернулли с $p' = 0{,}9998$ даёт то же число:

In [16]:
round(st.binom.pmf(9997, n, 1 - p), 4)

np.float64(0.1805)

* Рассмотрим п. **2б)** – **не** повреждено хотя бы $9997$ изделий

Это равносильно тому, что повреждено **не более** $3$ изделий:

$P_{10000}(m \leqslant 3) = P_{0,10000} + P_{1,10000} + P_{2,10000} + P_{3,10000}$

In [17]:
round(st.poisson.cdf(3, lam), 4)

np.float64(0.8571)

В учебнике получено $0{,}8572$: слагаемые там взяты из таблицы ($0{,}1353 + 0{,}2707 + 0{,}2707 + 0{,}1805$), и округление накапливается. Точное значение по Пуассону $0{,}857123\ldots$, по Бернулли $0{,}857141\ldots$.

**Сравнение приближения с точным ответом:**

In [18]:
rows = [
    ("P(m = 3)",   st.poisson.pmf(3, lam), st.binom.pmf(3, n, p)),
    ("P(m >= 3)",  st.poisson.sf(2, lam),  st.binom.sf(2, n, p)),
    ("P(m <= 3)",  st.poisson.cdf(3, lam), st.binom.cdf(3, n, p)),
]
print(f"{'событие':<12}{'Пуассон':>10}{'Бернулли':>10}")
for name, approx, exact in rows:
    print(f"{name:<12}{approx:>10.4f}{exact:>10.4f}")

событие        Пуассон  Бернулли
P(m = 3)        0.1804    0.1805
P(m >= 3)       0.3233    0.3233
P(m <= 3)       0.8571    0.8571


Формула Пуассона даёт совпадение до $4$ знаков – при таких $n$ и $p$ приближение очень точное.

**Ответ:** 1) а) $0{,}1804$; б) $0{,}3233$; 2) а) $0{,}1804$; б) $0{,}8571$.

## Пример 2.11
> По результатам проверок налоговыми инспекциями установлено, что в среднем каждое второе малое предприятие региона имеет нарушение финансовой дисциплины. Найти вероятность того, что из $1000$ зарегистрированных в регионе малых предприятий имеют нарушения финансовой дисциплины: а) $480$ предприятий; б) наивероятнейшее число предприятий; в) не менее $480$; г) от $480$ до $520$.

Здесь $p = 0{,}5$, $n = 1000$, и $npq = 250 \geqslant 20$ – условие применимости формул **Муавра–Лапласа** выполнено:

* локальная формула **(2.7)**: $P_{m,n} \approx \frac{f(x)}{\sqrt{npq}}$, где $x = \frac{m - np}{\sqrt{npq}}$;
* интегральная формула **(2.10)**: $P_n(a \leqslant m \leqslant b) \approx \frac{1}{2}\left[ \Phi(x_2) - \Phi(x_1) \right]$, где $x_1 = \frac{a - np}{\sqrt{npq}}$, $x_2 = \frac{b - np}{\sqrt{npq}}$.

In [19]:
n = 1000
p = 0.5
q = 1 - p

npq = n * p * q
sigma = np.sqrt(npq)   # корень из npq
print("np =", n * p, " npq =", npq, " sqrt(npq) =", round(sigma, 4))

np = 500.0  npq = 250.0  sqrt(npq) = 15.8114


* Рассмотрим п. **а)** – ровно $480$ предприятий

Сначала по формуле (2.9) находим $x$, затем применяем локальную формулу (2.7):

$x = \frac{480 - 1000 \cdot 0{,}5}{\sqrt{1000 \cdot 0{,}5 \cdot 0{,}5}} = -1{,}265, \qquad P_{480,1000} \approx \frac{f(-1{,}265)}{\sqrt{250}}$

In [20]:
x = (480 - n * p) / sigma
print("x =", round(x, 3))
round(st.norm.pdf(x) / sigma, 4)

x = -1.265


np.float64(0.0113)

Точное значение по формуле Бернулли для сравнения:

In [21]:
round(st.binom.pmf(480, n, p), 4)

np.float64(0.0113)

* Рассмотрим п. **б)** – **наивероятнейшее** число предприятий

Из двойного неравенства **(2.4)**: $np - q \leqslant m_0 \leqslant np + p$, т.е. $499{,}5 \leqslant m_0 \leqslant 500{,}5$, откуда целое $m_0 = 500$. При $m = np$ имеем $x = 0$:

$P_{500,1000} \approx \frac{f(0)}{\sqrt{250}}$

In [22]:
lo = round(n * p - q, 9)
hi = round(n * p + p, 9)
m0 = [k for k in range(n + 1) if lo <= k <= hi]
print("m0 =", m0)

x0 = (m0[0] - n * p) / sigma
print("x =", round(x0, 3))
round(st.norm.pdf(x0) / sigma, 4)

m0 = [500]
x = 0.0


np.float64(0.0252)

Проверим, что максимум биномиального ряда действительно при $m = 500$:

In [23]:
int(np.argmax(st.binom.pmf(np.arange(n + 1), n, p)))

500

* Рассмотрим п. **в)** – **не менее** $480$ предприятий

Событие «$m \geqslant 480$» – это $480 \leqslant m \leqslant 1000$. Для большого числа слагаемых применяем интегральную формулу (2.10):

$x_1 = \frac{480 - 500}{\sqrt{250}} = -1{,}265, \qquad x_2 = \frac{1000 - 500}{\sqrt{250}} = 31{,}6$

$P_{1000}(480 \leqslant m \leqslant 1000) \approx \frac{1}{2}\left[ \Phi(31{,}6) - \Phi(-1{,}265) \right] \approx \frac{1}{2}\left( 1 + 0{,}7941 \right) \approx 0{,}897$

In [24]:
x1 = (480 - n * p) / sigma
x2 = (1000 - n * p) / sigma
print("x1 =", round(x1, 3), " x2 =", round(x2, 1))

round(st.norm.cdf(x2) - st.norm.cdf(x1), 4)

x1 = -1.265  x2 = 31.6


np.float64(0.897)

* Рассмотрим п. **г)** – от $480$ до $520$ предприятий

Границы $480$ и $520$ **симметричны** относительно $np = 500$, поэтому удобно следствие **(2.13)**:

$P_{1000}(480 \leqslant m \leqslant 520) = P_{1000}\left( |m - 500| \leqslant 20 \right) \approx \Phi\left( \frac{20}{\sqrt{250}} \right) = \Phi(1{,}265) \approx 0{,}794$

Здесь $\Phi$ – функция Лапласа из учебника, т.е. $\Phi(t) = 2\,\texttt{norm.cdf}(t) - 1$.

In [25]:
t = 20 / sigma
print("t =", round(t, 3))
round(2 * st.norm.cdf(t) - 1, 4)

t = 1.265


np.float64(0.7941)

Тот же результат через разность значений `norm.cdf` на границах:

In [26]:
round(st.norm.cdf(t) - st.norm.cdf(-t), 4)

np.float64(0.7941)

**Сравнение приближений Муавра–Лапласа с точными ответами:**

In [27]:
rows = [
    ("а) m = 480",      st.norm.pdf(x) / sigma,                     st.binom.pmf(480, n, p)),
    ("б) m = 500",      st.norm.pdf(0) / sigma,                     st.binom.pmf(500, n, p)),
    ("в) m >= 480",     st.norm.cdf(x2) - st.norm.cdf(x1),          st.binom.sf(479, n, p)),
    ("г) 480<=m<=520",  st.norm.cdf(t) - st.norm.cdf(-t),           st.binom.cdf(520, n, p) - st.binom.cdf(479, n, p)),
]
print(f"{'событие':<16}{'Муавр-Лаплас':>14}{'Бернулли':>10}")
for name, approx, exact in rows:
    print(f"{name:<16}{approx:>14.4f}{exact:>10.4f}")

событие           Муавр-Лаплас  Бернулли
а) m = 480              0.0113    0.0113
б) m = 500              0.0252    0.0252
в) m >= 480             0.8970    0.9026
г) 480<=m<=520          0.7941    0.8052


Для вероятностей отдельных значений (пп. а, б) совпадение до $4$ знаков, а для сумм (пп. в, г) разница заметнее – во втором знаке. Дело в том, что интегральная формула заменяет сумму столбиков гистограммы площадью под кривой и «теряет» по половине крайних столбиков. Это исправляет **поправка на непрерывность** – границы сдвигаются на $0{,}5$ наружу: $479{,}5 \leqslant m \leqslant 520{,}5$.

In [28]:
a, b = 480 - 0.5, 520 + 0.5
round(st.norm.cdf((b - n * p) / sigma) - st.norm.cdf((a - n * p) / sigma), 4)

np.float64(0.8052)

С поправкой получаем $0{,}8049$ вместо $0{,}794$ – почти точное $0{,}8052$.

**Ответ:** а) $0{,}0113$; б) $m_0 = 500$, $P_{500,1000} \approx 0{,}0252$; в) $0{,}897$; г) $0{,}794$.